# ⚡ Kaggle Training AI

**RU:** Панель обучения своих моделей: датасет (ссылка/файл) → QLoRA-дообучение с живым
графиком → GGUF → импорт в Ollama (сразу доступно по `/v1`) → автозаливка на HuggingFace →
скачивание. Плюс всё из прошлой панели: терминал, статистика, туннель.

**EN:** A panel to train your own model: dataset (link/file) → QLoRA fine-tune with a live
loss chart → GGUF → Ollama import (instantly served on `/v1`) → auto-upload to HuggingFace
→ download. Plus everything from the previous panel: terminal, stats, tunnel.

### Запуск / How to start
1. **Settings → Internet → ON**, ускоритель **GPU** (T4).
2. Запусти ячейку ниже — всё настроится само. Открой **URL**, введи **пароль**.
3. Вкладка **Training** → датасет → имя модели → **Start training**.

### Фоновый режим (без 40-минутного простоя)
Save Version → **Save & Run All** — последняя ячейка (keep-alive) держит прогон до лимита (~9 ч GPU).
Впиши в CONFIG `password`, `cf_token`, `domain`, `pull_models` — тогда всё воспроизведётся само.

### HuggingFace
Впиши `hf_token` (hf_...) в CONFIG или в Settings панели — обученные модели зальются
в приватный репо `<user>/<имя-модели>` (GGUF + адаптер).


In [ ]:
#@title ⚡ Запустить панель обучения / Launch training panel { display-mode: "form" }
CONFIG = {
    "password": "",       # пароль панели (= API key); пусто = автогенерация
    "cf_token": "",       # Cloudflare tunnel token (свой домен, необязательно)
    "domain":  "",        # ваш домен панели и API (необязательно)
    "hf_token": "",       # HuggingFace токен для автозаливки моделей (hf_...)
    "port": 7860,
    "repo": "https://github.com/Jedafa/kaggle-training-ai.git",
    "install_ollama": True,
    "pull_models": [],    # модели ollama для авто-установки, напр. ["llama3.2:3b"]
}

import os, re, secrets, shlex, subprocess, sys, time, urllib.request
from pathlib import Path

HOME = Path.home()
BASE = HOME / ".kaggle-panel"
REPO_DIR = HOME / "kaggle-training-ai"
CF_BIN = BASE / "bin" / "cloudflared"
TUNNEL_LOG = BASE / "tunnel.log"

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def pgrep(pat):
    return subprocess.run(["pgrep", "-f", pat], capture_output=True).returncode == 0

# ---------- 1. repo ----------
if not (REPO_DIR / "server.py").exists():
    print("⏳ cloning panel repo ...")
    r = sh(f"git clone --depth 1 {CONFIG['repo']} {REPO_DIR}")
    if r.returncode != 0:
        print(r.stderr); raise SystemExit("clone failed — включите Internet в настройках")
else:
    sh(f"cd {REPO_DIR} && git pull --ff-only")
print("✅ repo:", REPO_DIR)

# ---------- 2. python deps ----------
PY = shlex.quote(sys.executable)
print("⏳ installing deps (transformers/peft/bitsandbytes — 2-4 мин) ...")
r = sh(f"{PY} -m pip install -q -r {shlex.quote(str(REPO_DIR / 'requirements.txt'))}")
if r.returncode != 0:
    r = sh(f"{PY} -m pip install -q --user -r {shlex.quote(str(REPO_DIR / 'requirements.txt'))}")
if r.returncode != 0:
    print(r.stdout[-800:], r.stderr[-800:])
    raise SystemExit("pip install failed")
print("✅ deps installed")

# ---------- 3. cloudflared ----------
if not CF_BIN.exists():
    print("⏳ downloading cloudflared ...")
    CF_BIN.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CF_BIN)
    CF_BIN.chmod(0o755)
print("✅ cloudflared:", CF_BIN)

# ---------- 4. password + .env ----------
PASSWORD = CONFIG["password"].strip() or secrets.token_urlsafe(9)
PORT = int(CONFIG["port"])
BASE.mkdir(parents=True, exist_ok=True)
(BASE / ".env").write_text(
    "PANEL_PORT=%s\nPANEL_PASSWORD=%s\nCF_TOKEN=%s\nCF_DOMAIN=%s\nTUNNEL_UUID=\nHF_TOKEN=%s\n"
    % (shlex.quote(str(PORT)), shlex.quote(PASSWORD),
       shlex.quote(CONFIG["cf_token"].strip()), shlex.quote(CONFIG["domain"].strip()),
       shlex.quote(CONFIG["hf_token"].strip())))

# ---------- 5. start panel ----------
subprocess.run(["pkill", "-f", r"python.*server\.py"], capture_output=True)
time.sleep(1)
env = dict(os.environ, PANEL_PASSWORD=PASSWORD, PANEL_PORT=str(PORT))
lf = open(BASE / "panel.log", "ab")
subprocess.Popen([sys.executable, "server.py"], cwd=REPO_DIR, env=env,
                 stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    print((BASE / "panel.log").read_text(errors="ignore")[-2000:])
    raise SystemExit("panel did not start")
print(f"✅ panel running on 127.0.0.1:{PORT}")

# ---------- 6. ollama + models фоном ----------
chain = ""
if CONFIG.get("install_ollama", True):
    chain += ("(apt-get install -y zstd >/dev/null 2>&1 || true); "
              "command -v ollama >/dev/null 2>&1 || { curl -fsSL https://ollama.com/install.sh | sh; }; ")
PULL = CONFIG.get("pull_models") or []
if PULL:
    chain += (f"pgrep -x ollama >/dev/null || {{ nohup ollama serve >{BASE}/ollama.log 2>&1 & sleep 2; }}; "
              + "; ".join(f"ollama pull {shlex.quote(m)}" for m in PULL))
if chain:
    subprocess.Popen(["bash", "-c", chain], stdout=lf, stderr=subprocess.STDOUT,
                     start_new_session=True)
    print("⏳ фоном ставится ollama/модели — лог: ~/.kaggle-panel/panel.log")

# ---------- 7. tunnel ----------
if not pgrep("cloudflared tunnel"):
    subprocess.run(["bash", str(BASE / "scripts" / "tunnel.sh"), "start"],
                   capture_output=True, text=True)

url = None
if not CONFIG["cf_token"].strip():
    print("⏳ waiting for trycloudflare URL ...")
    for _ in range(60):
        try:
            m = re.findall(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com",
                           TUNNEL_LOG.read_text(errors="ignore"))
        except OSError:
            m = []
        if m:
            url = m[-1]; break
        time.sleep(1)
    if not url:
        try:
            print("— tunnel.log tail: —")
            print(TUNNEL_LOG.read_text(errors="ignore")[-900:])
        except OSError:
            pass

# ---------- 8. summary ----------
try:
    from IPython.display import HTML, display
except ImportError:
    def display(x):
        print(x)
    class HTML(str):
        pass

DOMAIN = CONFIG["domain"].strip()
if url:
    panel_url = url
elif DOMAIN:
    panel_url = "https://" + DOMAIN
else:
    panel_url = None
ollama_url = (panel_url + "/ollama") if panel_url else None

print()
print("=" * 62)
print("  ⚡ KAGGLE TRAINING AI")
if panel_url:
    print("  🌐  Panel URL: ", panel_url)
elif CONFIG["cf_token"].strip():
    print("  🌐  Panel URL: named tunnel — открой домен из Cloudflare дашборда")
else:
    print("  ⚠️  туннель не поднялся — проверь Internet ON и запусти ячейку заново")
if ollama_url:
    print("  🦙  Ollama API:", ollama_url)
    print("      (OpenAI base:", ollama_url + "/v1 )")
    print("      ключ API = пароль панели (Authorization: Bearer)")
print("  🔑  PASSWORD:  ", PASSWORD)
print("=" * 62)

if panel_url:
    display(HTML(
        f'<div style="padding:14px;border-radius:12px;background:#0f1011;'
        f'font:600 15px sans-serif;color:#fff;line-height:1.7">⚡ Панель: '
        f'<a href="{panel_url}" style="color:#e4f222">{panel_url}</a><br>'
        f'🦙 Ollama API: <code style="color:#02b8cc">{ollama_url}</code><br>'
        f'🔑 пароль / API key: <code style="color:#e4f222">{PASSWORD}</code></div>'))
print("\nℹ️ Дальше: вкладка Training → датасет → имя модели → Start training. "
      "Готовая модель появится в Ollama (Models → chat) и на HuggingFace.")


---
### Полезное
- Потеряли ссылку — запусти ячейку ниже.
- Токен HuggingFace и настройки туннеля меняются и в панели: **Settings**.


In [ ]:
#@title 🔎 Показать URL и пароль ещё раз { display-mode: "form" }
import re
from pathlib import Path
try:
    from IPython.display import HTML, display
except ImportError:
    def display(x):
        print(x)
    class HTML(str):
        pass
BASE = Path.home() / ".kaggle-panel"
env = {}
if (BASE / ".env").exists():
    for line in (BASE / ".env").read_text().splitlines():
        if "=" in line and not line.strip().startswith("#"):
            k, _, v = line.partition("="); env[k.strip()] = v.strip().strip('"')
url = None
try:
    m = re.findall(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com",
                   (BASE / "tunnel.log").read_text(errors="ignore"))
    url = m[-1] if m else None
except OSError:
    pass
domain = env.get("CF_DOMAIN", "").strip().strip('"')
panel_url = url or (("https://" + domain) if domain else None)
ollama_url = (panel_url + "/ollama") if panel_url else None
html = f'<div style="padding:14px;border-radius:12px;background:#0f1011;font:600 15px sans-serif;color:#fff;line-height:1.7">'
html += "⚡ " + (f'<a href="{panel_url}" style="color:#e4f222">{panel_url}</a>' if panel_url else "туннель не запущен — Start tunnel в панели")
if ollama_url:
    html += f'<br>🦙 Ollama API: <code style="color:#02b8cc">{ollama_url}</code> (ключ = пароль)'
html += f'<br>🔑 пароль: <code style="color:#e4f222">{env.get("PANEL_PASSWORD","?")}</code></div>'
display(HTML(html))


In [ ]:
#@title ⏳ Keep-alive — держит сессию живой (для Save & Run All) { display-mode: "form" }
# Для фонового прогона (Save Version → Save & Run All) эта ячейка должна быть ПОСЛЕДНЕЙ:
# она не даёт Commit'у завершиться, пока не упрёшься в лимит (~9 ч GPU / ~12 ч CPU).
import os, time, urllib.request
env = {}
p = os.path.expanduser("~/.kaggle-panel/.env")
if os.path.exists(p):
    for line in open(p, errors="ignore"):
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            k, _, v = line.partition("="); env[k.strip()] = v.strip().strip('"')
PORT = int(env.get("PANEL_PORT", "7860") or 7860)
print(f"⏳ keep-alive запущен — панель на 127.0.0.1:{PORT}")
i = 0
while True:
    time.sleep(60)
    i += 1
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=5)
    except Exception:
        pass
    if i % 30 == 0:
        print(f"alive: {i} мин")
